In [1]:
import math
import sys
from pathlib import Path

_here = Path().resolve()
for _p in [_here, *_here.parents]:
    _src = _p / "src"
    if (_src / "qudits_on_qubits" / "__init__.py").is_file():
        repo_root = _p
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
else:
    raise ImportError(
        "qudits_on_qubits repo root not found; run the notebook from notebooks/ or the repo root"
    )

from qiskit.quantum_info import Statevector, Operator, partial_trace, SparsePauliOp
from qiskit import qpy, QuantumCircuit
import numpy as np
from qiskit.synthesis import TwoQubitWeylDecomposition
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, Batch

from qudits_on_qubits import create_ame_circuit, generate_b_ame
from sympy.functions.combinatorial.numbers import legendre_symbol
from IPython.display import display, Math
from itertools import product
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from igraph import Graph, plot
import matplotlib.pyplot as plt

In [2]:
selected_candidate = "monomial_full__sup013_P102_ph112"
artifact_circuit_dir = repo_root / "artifacts" / "direct_basis_runs" / "selected_best" / "two_qutrit" / selected_candidate
legacy_circuit_dir = repo_root.parent / "QuditsOnQubits_legacy" / "basis_direct_encoding_benchmarks" / "quantum_circuits" / "two_qutrit" / selected_candidate

for circuit_dir in (artifact_circuit_dir, legacy_circuit_dir):
    if (circuit_dir / "graph_state_direct_basis.qpy").is_file():
        break
else:
    raise FileNotFoundError(
        "Circuit artifacts not found. Checked:\n"
        f"- {artifact_circuit_dir}\n"
        f"- {legacy_circuit_dir}"
    )

print(f"Loading circuits from: {circuit_dir}")

with (circuit_dir / "graph_state_direct_basis.qpy").open("rb") as f:
    testqc = qpy.load(f)[0]

with (circuit_dir / "graph_state_direct_basis_transpiled.qpy").open("rb") as f:
    qcsuptrans = qpy.load(f)[0]

with (circuit_dir / "F3_W.qpy").open("rb") as f:
    F3sup = qpy.load(f)[0]

Esup = np.load(circuit_dir / "E.npy")

In [3]:
qtr2, graph2 = create_ame_circuit(2, 3)

In [4]:
from qudits_on_qubits.bell_measurements.sampler_circuits import build_sampler_circuits_from_graph, build_sampler_circuits_for_candidate
from qudits_on_qubits.bell_measurements.basis import canonical_Ez

In [5]:
E = canonical_Ez()

In [6]:
sampler_circuits, metadata = build_sampler_circuits_for_candidate(candidate="two_qutrit", state_circuit=qtr2, E=E)

In [7]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2

In [8]:
from qudits_on_qubits.bell_measurements.sampler_circuits import run_sampler_circuits_to_counts_by_setting

from qudits_on_qubits.bell_measurements.sampler_circuits import decoding_kwargs_from_metadata

In [9]:
counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(sampler_circuits, metadata, shots=1024*50)

In [10]:
from qudits_on_qubits.bell_measurements.postprocessing import compute_bell_value_from_counts

In [11]:
bell_value = compute_bell_value_from_counts(counts_by_setting, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))

In [12]:
bell_value

(6.004877428697462-7.431555371084642e-15j)

In [13]:
sampler_circuits, metadata = build_sampler_circuits_for_candidate(candidate="two_qutrit", state_circuit=testqc, E=Esup)
counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(sampler_circuits, metadata, shots=1024*50)
bell_value = compute_bell_value_from_counts(counts_by_setting, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(5.996084946030004-7.389922007661198e-15j)